# Bangladeshi Traffic Sign Classification — Exploration & Training Notebook

This notebook walks through the same pipeline as the `src/` scripts, but interactively — useful for exploring the real dataset, visualizing class distribution, and sanity-checking augmentation before running full training with `python -m src.train`.

**Before running this notebook:** download a real Bangladeshi traffic sign dataset and place it at `data/raw/<ClassName>/<image files>` (see README.md for dataset sources).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src import config
from src.data_preprocessing import verify_raw_dataset, split_dataset
from src.data_augmentation import get_datasets, get_augmentation_layer
import matplotlib.pyplot as plt
import numpy as np

## 1. Verify the real dataset and check class balance

In [ ]:
classes = verify_raw_dataset()
counts = {c: len(os.listdir(os.path.join(config.RAW_DATA_DIR, c))) for c in classes}

plt.figure(figsize=(8, 4))
plt.bar(counts.keys(), counts.values())
plt.xticks(rotation=45, ha='right')
plt.ylabel('Number of real images')
plt.title('Class distribution in raw dataset')
plt.tight_layout()
plt.show()
print(counts)

## 2. Split into train/val/test (real images only, no synthetic data)

In [ ]:
summary = split_dataset()
summary

## 3. Preview a few real training images before/after augmentation

In [ ]:
train_ds, val_ds, test_ds = get_datasets()
aug_layer = get_augmentation_layer()

images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i in range(5):
    axes[0, i].imshow(images[i].numpy())
    axes[0, i].set_title('augmented (train pipeline)')
    axes[0, i].axis('off')
    axes[1, i].imshow(np.clip(aug_layer(images[i:i+1], training=True)[0].numpy(), 0, 1))
    axes[1, i].set_title('re-augmented sample')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## 4. Train from the notebook (optional — or just run `python -m src.train` from a terminal)

In [ ]:
from src.train import main as train_main
train_main(['custom_cnn', 'mobilenetv2', 'efficientnetb0'])

## 5. Evaluate the best model

In [ ]:
from src.evaluate import evaluate_model
metrics = evaluate_model('best_model')
metrics